# Notebook 07 — Pipeline RAG Completo

**Settimana:** 7
**Blocco:** 4.4 — Pipeline retrieval + generazione
**Tempo stimato:** 3,5-4 ore

## Cosa costruirai

Al termine di questo notebook avrai un chatbot RAG funzionante su qualsiasi corpus di documenti:
- Pipeline di ingestione modulare (caricare → pulire → chunkare → indicizzare)
- Pipeline di query completa (embed → retrieve → prompt → generare)
- Query rewriting e multi-query retrieval
- Gestione del caso "non ci sono informazioni rilevanti"
- **Sfida finale:** chatbot RAG sulla documentazione di FastAPI con cronologia della conversazione

## Prerequisiti

Settimane 5 e 6 completate. Chroma e le funzioni di embedding disponibili.

```bash
pip install chromadb langchain-text-splitters openai anthropic pymupdf requests python-dotenv
```

---
## Cella 0 — Setup: import e client

In [ ]:
import os
import re
import json
import hashlib
import datetime
import requests
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import Literal
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

cliente_openai    = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
cliente_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
chroma_cliente    = chromadb.PersistentClient(path="./chroma_db")

MODELO_EMBEDDING  = "text-embedding-3-small"
MODELO_GENERACION = "gpt-4o-mini"

# ── Helper di embedding (dalle settimane precedenti) ─────────────────────────
def generar_embedding(texto: str) -> list[float]:
    r = cliente_openai.embeddings.create(model=MODELO_EMBEDDING, input=texto)
    return r.data[0].embedding

def generar_embeddings_batch(textos: list[str]) -> np.ndarray:
    r = cliente_openai.embeddings.create(model=MODELO_EMBEDDING, input=textos)
    return np.array([item.embedding for item in sorted(r.data, key=lambda x: x.index)])

# ── Helper di generazione ─────────────────────────────────────────────────────
def completar(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.1,
    max_tokens: int = 1024
) -> str:
    """Chiamata semplice all'LLM, restituisce il testo."""
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente_openai.chat.completions.create(
        model=MODELO_GENERACION,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return r.choices[0].message.content

Path("datos_rag").mkdir(exist_ok=True)
print("Setup completato.")

---
## Cella 1 — Pipeline di ingestione modulare

Costruisci ogni fase come una funzione indipendente.
Alla fine concatenale in `ingestar_documentos()`.

Usiamo la documentazione di FastAPI (scaricata da GitHub in Markdown)
perché è un corpus reale, ben strutturato e in inglese/spagnolo.

In [ ]:
# ── Fonti dei documenti ───────────────────────────────────────────────────────
# Scarichiamo pagine della doc di FastAPI in testo semplice da Wikipedia
# (alternativa gratuita senza bisogno di scraping complesso)

FUENTES = [
    {"titulo": "FastAPI",                  "tipo": "framework"},
    {"titulo": "Representational state transfer", "tipo": "concepto"},
    {"titulo": "Pydantic",                 "tipo": "libreria"},
    {"titulo": "JSON Web Token",           "tipo": "seguridad"},
    {"titulo": "Docker (software)",        "tipo": "infraestructura"},
    {"titulo": "PostgreSQL",               "tipo": "base_datos"},
]

# ── Fase 1: Loader — carica il testo da Wikipedia ────────────────────────────

def cargar_wikipedia(titulo: str) -> str:
    """Scarica il testo completo di un articolo di Wikipedia in inglese."""
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query", "format": "json",
        "titles": titulo, "prop": "extracts",
        "explaintext": True, "exsectionformat": "plain"
    }
    try:
        resp   = requests.get(url, params=params, timeout=10)
        pages  = resp.json()["query"]["pages"]
        texto  = list(pages.values())[0].get("extract", "")
        return texto
    except Exception as e:
        print(f"  Errore nel caricare '{titulo}': {e}")
        return ""

# ── Fase 2: Cleaner — pulizia di base ────────────────────────────────────────

def limpiar(texto: str) -> str:
    """Rimuove il rumore: caratteri di controllo, spazi multipli, sezioni vuote."""
    texto = re.sub(r"[\x00-\x08\x0b\x0e-\x1f\x7f]", "", texto)
    texto = re.sub(r" +",    " ",   texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"== References ==.*", "", texto, flags=re.DOTALL)  # rimuove i riferimenti di Wikipedia
    return texto.strip()

# ── Fase 3: Chunker ───────────────────────────────────────────────────────────

_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def chunkear(texto: str) -> list[str]:
    """Divide il testo in chunk di 500 caratteri con 100 di sovrapposizione."""
    return _splitter.split_text(texto)

# ── Fase 4: Indexer — salva in Chroma ────────────────────────────────────────

def indexar_en_coleccion(
    coleccion: chromadb.Collection,
    chunks: list[str],
    metadatos_base: dict,
    batch_size: int = 50
) -> int:
    """Genera embedding e indicizza i chunk. Restituisce il numero di chunk aggiunti."""
    total = len(chunks)
    for inicio in range(0, total, batch_size):
        lote      = chunks[inicio:min(inicio + batch_size, total)]
        ids       = [
            f"{metadatos_base['fuente']}_{i}"
            for i in range(inicio, inicio + len(lote))
        ]
        embeddings = generar_embeddings_batch(lote).tolist()
        metas      = [
            {**metadatos_base, "chunk_index": i, "total_chunks": total, "n_chars": len(c)}
            for i, c in enumerate(lote, inicio)
        ]
        coleccion.upsert(ids=ids, documents=lote, embeddings=embeddings, metadatas=metas)
    return total

# ── Pipeline di ingestione completa ──────────────────────────────────────────

def ingestar_documentos(
    fuentes: list[dict],
    nombre_coleccion: str = "rag_docs"
) -> chromadb.Collection:
    """Orchestra Loader → Cleaner → Chunker → Indexer per tutte le fonti."""
    try:
        chroma_cliente.delete_collection(nombre_coleccion)
    except Exception:
        pass
    coleccion = chroma_cliente.create_collection(
        name=nombre_coleccion,
        metadata={"hnsw:space": "cosine"}
    )

    total_chunks = 0
    for fuente in fuentes:
        titulo = fuente["titulo"]
        print(f"\nIngestione: '{titulo}'")

        texto    = cargar_wikipedia(titulo)
        if not texto:
            print(f"  Saltato (nessun contenuto)")
            continue

        texto    = limpiar(texto)
        chunks   = chunkear(texto)
        n        = indexar_en_coleccion(
            coleccion, chunks,
            metadatos_base={
                "fuente":        titulo,
                "tipo":          fuente.get("tipo", "general"),
                "fecha_ingesta": datetime.date.today().isoformat(),
                "idioma":        "en"
            }
        )
        print(f"  → {n} chunk indicizzati")
        total_chunks += n

    print(f"\nIngestione completata. Totale: {total_chunks} chunk in '{nombre_coleccion}'")
    return coleccion


# ── Esegue l'ingestione ───────────────────────────────────────────────────────
coleccion_rag = ingestar_documentos(FUENTES, nombre_coleccion="rag_docs")
print(f"\nDocumenti nella collezione: {coleccion_rag.count()}")

---
## Cella 2 — Pipeline di query di base

Costruisci le tre funzioni della pipeline di query:
`recuperar()`, `construir_prompt()`, `generar()`.
Poi concatenale in `responder()` e testa con diverse domande.

In [ ]:
UMBRAL_RELEVANCIA = 0.68

SYSTEM_RAG = """You are a technical documentation assistant.
Answer questions using ONLY the information provided in the context.
If the answer is not found in the context, say: 'I don't have enough information in the available documentation to answer this.'
Do not use external knowledge. Always cite the source at the end of your answer."""

# ── Funzione 1: recuperar ─────────────────────────────────────────────────────

def recuperar(
    query: str,
    coleccion: chromadb.Collection,
    k: int = 4,
    filtro: dict | None = None
) -> list[dict]:
    """
    Genera l'embedding della query e recupera i k chunk più simili.
    Restituisce una lista di dict con testo, similarità e metadati.
    Nota: Chroma con metrica coseno restituisce la distanza (0=identico, 2=opposto).
          La convertiamo in similarità: sim = 1 - distanza.
    """
    vector = generar_embedding(query)
    kwargs = {
        "query_embeddings": [vector],
        "n_results":        min(k, coleccion.count()),
        "include":          ["documents", "distances", "metadatas"]
    }
    if filtro:
        kwargs["where"] = filtro

    resp = coleccion.query(**kwargs)

    resultados = []
    for doc, dist, meta in zip(
        resp["documents"][0],
        resp["distances"][0],
        resp["metadatas"][0]
    ):
        resultados.append({
            "texto":     doc,
            "similitud": round(1 - dist, 4),
            "meta":      meta
        })
    return resultados


# ── Funzione 2: construir_prompt ─────────────────────────────────────────────

def construir_prompt(query: str, chunks: list[dict]) -> str:
    """
    Formatta i chunk recuperati come contesto numerato.
    Include lo score di rilevanza e i metadati in modo che l'LLM possa citare.
    """
    bloques = []
    for i, chunk in enumerate(chunks, 1):
        meta  = chunk["meta"]
        fuente = meta.get("fuente", "desconocida")
        score  = chunk["similitud"]
        bloques.append(
            f"[Context {i} | Source: {fuente} | Relevance: {score:.2f}]\n"
            f"{chunk['texto']}"
        )

    contexto = "\n\n---\n\n".join(bloques)

    return f"""CONTEXT:
{contexto}

QUESTION: {query}

ANSWER (based only on the context above):"""


# ── Funzione 3: orchestratore ────────────────────────────────────────────────

@dataclass
class RespuestaRAG:
    pregunta:    str
    respuesta:   str
    fuentes:     list[dict]
    n_chunks:    int
    respondida:  bool    # False se non ci sono stati chunk sufficientemente rilevanti

    def mostrar(self) -> None:
        estado = "✓" if self.respondida else "⚠ Informazioni insufficienti"
        print(f"\n{estado}")
        print(f"Domanda:   {self.pregunta}")
        print(f"Risposta:  {self.respuesta}")
        if self.fuentes:
            fuentes_unicas = list({f.get("fuente", "?") for f in self.fuentes})
            print(f"Fonti ({self.n_chunks} chunk): {', '.join(fuentes_unicas)}")


def responder(
    pregunta: str,
    coleccion: chromadb.Collection,
    k: int = 4,
    umbral: float = UMBRAL_RELEVANCIA,
    filtro: dict | None = None
) -> RespuestaRAG:
    """Pipeline completa: retrieve → filtrare per soglia → costruire il prompt → generare."""

    # Recupera e filtra per rilevanza
    chunks_todos    = recuperar(pregunta, coleccion, k=k, filtro=filtro)
    chunks_relevantes = [c for c in chunks_todos if c["similitud"] >= umbral]

    if not chunks_relevantes:
        return RespuestaRAG(
            pregunta=pregunta,
            respuesta="I don't have enough information in the available documentation to answer this.",
            fuentes=[],
            n_chunks=0,
            respondida=False
        )

    prompt    = construir_prompt(pregunta, chunks_relevantes)
    respuesta = completar(
        mensajes=[{"role": "user", "content": prompt}],
        system=SYSTEM_RAG
    )

    return RespuestaRAG(
        pregunta=pregunta,
        respuesta=respuesta,
        fuentes=[c["meta"] for c in chunks_relevantes],
        n_chunks=len(chunks_relevantes),
        respondida=True
    )


# ── Test della pipeline di base ───────────────────────────────────────────────
PREGUNTAS_TEST = [
    "What is FastAPI and what are its main features?",
    "How does JWT authentication work?",
    "What is Docker and why is it used for deployment?",
    "What is the meaning of life?",   # domanda fuori dal corpus — deve fallire correttamente
]

for p in PREGUNTAS_TEST:
    r = responder(p, coleccion_rag)
    r.mostrar()
    print()

**Osservazioni** (modifica questa cella):

- Le risposte citano correttamente la fonte?
- La domanda fuori dal corpus ha restituito il messaggio di "informazioni insufficienti"?
- C'è qualche risposta che contiene informazioni NON presenti nel contesto?

---
## Cella 3 — Query rewriting e multi-query retrieval

Due tecniche che migliorano il retrieval prima di chiamare l'LLM.
Confronta i chunk recuperati con e senza ciascuna tecnica.

In [ ]:
# ── Tecnica 1: query rewriting ────────────────────────────────────────────────

def reescribir_query(pregunta: str) -> str:
    """
    Riformula la domanda dell'utente per ottimizzarla per la ricerca semantica.
    Utile quando l'utente usa un linguaggio colloquiale o ambiguo.
    """
    prompt = f"""Rewrite the following question as an optimized semantic search query.
The rewritten query should:
- Be more specific and technical
- Remove conversational filler words
- Use key technical terms
- Be in declarative form if more effective

Original question: {pregunta}
Optimized query (only the query, no explanation):"""

    return completar(
        mensajes=[{"role": "user", "content": prompt}],
        temperature=0.0
    ).strip()


# ── Tecnica 2: multi-query retrieval ─────────────────────────────────────────

def generar_queries_alternativas(pregunta: str, n: int = 3) -> list[str]:
    """
    Genera N riformulazioni distinte della stessa domanda.
    Ognuna affronta il concetto da un'angolazione diversa.
    """
    prompt = f"""Generate {n} different search queries to find information about the same topic as this question.
Each query should approach the concept from a different angle.
Return only the queries, one per line, no numbering or explanation.

Question: {pregunta}"""

    respuesta = completar(
        mensajes=[{"role": "user", "content": prompt}],
        temperature=0.4
    )
    return [q.strip() for q in respuesta.strip().split("\n") if q.strip()][:n]


def recuperar_multi_query(
    pregunta: str,
    coleccion: chromadb.Collection,
    k_por_query: int = 3,
    n_queries: int = 3
) -> list[dict]:
    """
    Genera N query alternative, recupera con ciascuna e deduplica.
    Restituisce i chunk unici ordinati per massima similarità.
    """
    queries_alt = generar_queries_alternativas(pregunta, n=n_queries)
    todas_queries = [pregunta] + queries_alt

    print(f"  Query generate:")
    for q in todas_queries:
        print(f"    - {q}")

    # Recupera con tutte le query
    vistos   = set()   # per deduplicare per testo
    unicos   = []
    for q in todas_queries:
        chunks = recuperar(q, coleccion, k=k_por_query)
        for c in chunks:
            clave = c["texto"][:100]   # fingerprint del chunk
            if clave not in vistos:
                vistos.add(clave)
                unicos.append(c)

    # Ordina per similarità decrescente e prende i top-k globali
    unicos.sort(key=lambda x: -x["similitud"])
    return unicos[:k_por_query * 2]   # top 2×k chunk unici


# ── Confronto: base vs rewriting vs multi-query ───────────────────────────────

QUERY_COLOQUIAL = "hey, how do you keep APIs safe from bad actors?"

print(f"Query originale: '{QUERY_COLOQUIAL}'")
print("=" * 60)

# Base: query così com'è
print("\n[1] Pipeline DI BASE (senza rewriting):")
chunks_basico = recuperar(QUERY_COLOQUIAL, coleccion_rag, k=3)
for c in chunks_basico:
    print(f"  {c['similitud']:.4f} | [{c['meta']['fuente'][:20]}] {c['texto'][:70]}...")

# Con rewriting
print("\n[2] Con QUERY REWRITING:")
query_reescrita = reescribir_query(QUERY_COLOQUIAL)
print(f"  Query riscritta: '{query_reescrita}'")
chunks_rewrite = recuperar(query_reescrita, coleccion_rag, k=3)
for c in chunks_rewrite:
    print(f"  {c['similitud']:.4f} | [{c['meta']['fuente'][:20]}] {c['texto'][:70]}...")

# Multi-query
print("\n[3] Con MULTI-QUERY RETRIEVAL:")
chunks_multi = recuperar_multi_query(QUERY_COLOQUIAL, coleccion_rag)
print(f"  Totale chunk unici recuperati: {len(chunks_multi)}")
for c in chunks_multi[:4]:
    print(f"  {c['similitud']:.4f} | [{c['meta']['fuente'][:20]}] {c['texto'][:70]}...")

**Osservazioni** (modifica questa cella):

- La query riscritta ha migliorato gli score di similarità?
- Il multi-query ha recuperato chunk che quello base non trovava? Quali?
- Quando vale la pena il costo extra del multi-query (2 chiamate LLM in più)?

---
## Cella 4 — Rispondere con multi-query e verificare la non-risposta

Collega il multi-query alla pipeline completa e testa il comportamento
di fronte a domande fuori dal corpus.

In [ ]:
def responder_avanzado(
    pregunta: str,
    coleccion: chromadb.Collection,
    usar_rewriting: bool = True,
    usar_multi_query: bool = False,
    k: int = 4,
    umbral: float = UMBRAL_RELEVANCIA,
    verbose: bool = False
) -> RespuestaRAG:
    """
    Pipeline avanzata con rewriting e/o multi-query opzionali.
    """
    query_efectiva = pregunta

    # Passo opzionale: riscrivere la query
    if usar_rewriting:
        query_efectiva = reescribir_query(pregunta)
        if verbose:
            print(f"  Query riscritta: '{query_efectiva}'")

    # Recupero
    if usar_multi_query:
        chunks_todos = recuperar_multi_query(query_efectiva, coleccion, k_por_query=k)
    else:
        chunks_todos = recuperar(query_efectiva, coleccion, k=k)

    chunks_relevantes = [c for c in chunks_todos if c["similitud"] >= umbral]

    if verbose:
        print(f"  Chunk recuperati: {len(chunks_todos)}, rilevanti (>={umbral}): {len(chunks_relevantes)}")

    if not chunks_relevantes:
        return RespuestaRAG(
            pregunta=pregunta,
            respuesta="I don't have enough information in the available documentation to answer this.",
            fuentes=[], n_chunks=0, respondida=False
        )

    prompt    = construir_prompt(pregunta, chunks_relevantes)
    respuesta = completar(
        mensajes=[{"role": "user", "content": prompt}],
        system=SYSTEM_RAG
    )

    return RespuestaRAG(
        pregunta=pregunta,
        respuesta=respuesta,
        fuentes=[c["meta"] for c in chunks_relevantes],
        n_chunks=len(chunks_relevantes),
        respondida=True
    )


# ── Test del caso di non-risposta: domande fuori dal corpus ───────────────────

print("=" * 60)
print("TEST: DOMANDE FUORI DAL CORPUS")
print("=" * 60)

PREGUNTAS_FUERA = [
    "What is the current price of Bitcoin?",
    "Who won the last FIFA World Cup?",
    "What is the best pizza recipe?",
]

for p in PREGUNTAS_FUERA:
    r = responder_avanzado(p, coleccion_rag, verbose=True)
    icono = "✓ Correttamente rifiutata" if not r.respondida else "✗ ERRORE: ha risposto senza informazioni"
    print(f"\n{icono}: '{p}'")
    if r.respondida:
        print(f"  Risposta (non dovrebbe esistere): {r.respuesta[:100]}...")

# ── Confronto rewriting vs senza rewriting su una domanda reale ───────────────

print("\n" + "=" * 60)
print("CONFRONTO: con e senza rewriting")
print("=" * 60)

PREGUNTA_COLOQUIAL = "I heard Pydantic is useful, what does it do exactly?"

r_sin = responder(PREGUNTA_COLOQUIAL, coleccion_rag, k=3)
r_con = responder_avanzado(PREGUNTA_COLOQUIAL, coleccion_rag, usar_rewriting=True, k=3, verbose=True)

print(f"\nRisposta SENZA rewriting ({r_sin.n_chunks} chunk):")
print(f"  {r_sin.respuesta[:200]}...")
print(f"\nRisposta CON rewriting ({r_con.n_chunks} chunk):")
print(f"  {r_con.respuesta[:200]}...")

---
## Sfida finale — Chatbot RAG con cronologia della conversazione

Estendi la pipeline affinché l'assistente ricordi la conversazione precedente.
La cronologia permette di fare domande di follow-up: "e a cosa serve?"
oppure "dammi più dettagli sul terzo punto".

Leggi lo scheletro e completa i `# TODO`.

In [ ]:
@dataclass
class ChatRAG:
    """
    Chatbot RAG con cronologia della conversazione.

    La cronologia serve a due cose:
    1. Contesto conversazionale per l'LLM (ricorda i turni precedenti)
    2. Arricchire la query di retrieval con il contesto precedente
    """
    coleccion:       chromadb.Collection
    k:               int   = 4
    umbral:          float = 0.65
    max_historial:   int   = 6    # massimo di turni da mantenere (per controllare i token)
    historial:       list  = field(default_factory=list)
    fuentes_sesion:  list  = field(default_factory=list)

    def _construir_query_con_contexto(self, pregunta: str) -> str:
        """
        Arricchisce la query con il contesto degli ultimi turni.
        Utile per domande di follow-up come "e a cosa serve?".
        """
        if not self.historial:
            return pregunta

        # Prende solo gli ultimi 2 turni per non sovraccaricare
        contexto_reciente = self.historial[-4:]   # 2 turni = 4 messaggi (user+assistant)
        resumen_contexto  = " ".join(
            m["content"][:100] for m in contexto_reciente if m["role"] == "assistant"
        )

        if len(resumen_contexto) < 20:   # non c'è contesto utile
            return pregunta

        return f"{pregunta} (context: {resumen_contexto[:200]})"

    def responder(self, pregunta: str, verbose: bool = False) -> str:
        """
        Risponde a una domanda usando RAG + cronologia della conversazione.

        Flusso:
        1. Arricchisce la query con la cronologia per migliorare il retrieval
        2. Recupera i chunk rilevanti
        3. Costruisce il prompt includendo la cronologia della conversazione
        4. Genera la risposta
        5. Aggiunge il turno alla cronologia
        """
        # TODO: implementa il flusso completo
        #
        # Passo 1: arricchisci la query
        # query_retrieval = self._construir_query_con_contexto(pregunta)
        #
        # Passo 2: recupera i chunk
        # chunks = recuperar(query_retrieval, self.coleccion, k=self.k)
        # chunks_relevantes = [c for c in chunks if c["similitud"] >= self.umbral]
        #
        # Passo 3: se non ci sono chunk rilevanti, restituisci il messaggio standard
        #
        # Passo 4: costruisci il prompt con il contesto RAG
        # (usa construir_prompt() della cella precedente)
        #
        # Passo 5: chiama l'LLM con la cronologia della conversazione
        # Suggerimento: i messaggi inviati all'LLM devono essere:
        #   [system] + historial[-max_historial:] + [user: prompt_con_contexto]
        # NON includere la cronologia nel prompt di retrieval,
        # solo nella chiamata all'LLM.
        #
        # Passo 6: salva nella cronologia
        # self.historial.append({"role": "user",      "content": pregunta})
        # self.historial.append({"role": "assistant", "content": texto_respuesta})
        # self.fuentes_sesion.extend([c["meta"] for c in chunks_relevantes])

        raise NotImplementedError("Implementa il metodo responder()")

    def mostrar_historial(self) -> None:
        """Mostra la cronologia della conversazione."""
        print(f"\n{'='*60}")
        print(f"Cronologia ({len(self.historial)//2} turni)")
        print(f"{'='*60}")
        for m in self.historial:
            rol    = "Tu" if m["role"] == "user" else "Assistente"
            print(f"\n[{rol}]")
            print(f"  {m['content'][:200]}{'...' if len(m['content']) > 200 else ''}")

    def resetear(self) -> None:
        """Pulisce la cronologia per iniziare una nuova sessione."""
        self.historial      = []
        self.fuentes_sesion = []
        print("Cronologia azzerata.")


# ── Test del chatbot con domande di follow-up ─────────────────────────────────
# Decommenta quando hai implementato ChatRAG.responder()

# chat = ChatRAG(coleccion=coleccion_rag, k=4, umbral=0.65)

# DIALOGO = [
#     "What is FastAPI?",
#     "What are its main advantages over Flask?",   # domanda di follow-up
#     "How does it handle data validation?",         # continua a parlare di FastAPI
#     "And what library does it use for that?",       # riferimento implicito a Pydantic
#     "What is the best programming language?",       # fuori dal corpus — deve fallire correttamente
# ]

# for turno, pregunta in enumerate(DIALOGO, 1):
#     print(f"\n[Turno {turno}] Domanda: {pregunta}")
#     print("-" * 50)
#     respuesta = chat.responder(pregunta, verbose=True)
#     print(f"Risposta: {respuesta[:300]}..." if len(respuesta) > 300 else f"Risposta: {respuesta}")

# chat.mostrar_historial()

### Estensioni opzionali

1. **Streaming nella sfida:** modifica `ChatRAG.responder()` in modo che usi lo streaming e mostri i token in tempo reale usando `completar_stream()` della settimana 3.

2. **Compressione della cronologia:** quando la cronologia supera `max_historial` turni, invece di scartarli, chiedi all'LLM di riassumerli in 2-3 frasi e usa quel riassunto come contesto.

3. **Interfaccia Streamlit:** costruisci un'interfaccia web minima con Streamlit che avvolga `ChatRAG`. Sarà la base del Progetto 1 della settimana 12.

In [ ]:
# Estensione 3: scheletro di interfaccia Streamlit
# Salva questo come app_rag.py ed eseguilo con: streamlit run app_rag.py

STREAMLIT_APP = '''
import streamlit as st
import os
import chromadb
from dotenv import load_dotenv

# Importa qui le tue funzioni dal notebook
# from rag_pipeline import ChatRAG, ingestar_documentos, ...

load_dotenv()

st.set_page_config(page_title="RAG Chatbot", page_icon="🤖")
st.title("📚 RAG Chatbot")
st.caption("Fai una domanda sulla documentazione indicizzata")

# Stato della sessione
if "mensajes" not in st.session_state:
    st.session_state.mensajes = []
if "chat" not in st.session_state:
    chroma_cli = chromadb.PersistentClient(path="./chroma_db")
    coleccion  = chroma_cli.get_collection("rag_docs")
    # st.session_state.chat = ChatRAG(coleccion=coleccion)

# Mostra la cronologia
for msg in st.session_state.mensajes:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Input dell'utente
if prompt := st.chat_input("Scrivi la tua domanda..."):
    st.session_state.mensajes.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Sto cercando nella documentazione..."):
            # respuesta = st.session_state.chat.responder(prompt)
            respuesta = "[Implementa prima ChatRAG]"
        st.markdown(respuesta)

    st.session_state.mensajes.append({"role": "assistant", "content": respuesta})
'''

with open("app_rag.py", "w") as f:
    f.write(STREAMLIT_APP)

print("app_rag.py creato. Per eseguirlo:")
print("  pip install streamlit")
print("  streamlit run app_rag.py")

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 07 — Settimana 7 del Piano di Formazione AI Engineering*